This notebook looks into variables that are highly correlated with identified variables of interest from 1ModelSetting.ipynb as we remove highly correlated variables with spearman correlation $\rho > 0.7$ in the data cleaning pipeline.

The variables of interest are:
- cve rate(+)
- no high school diploma rate(+)
- mammogram uptake in the past 2 years for women aged 40 and above rate(-)
- percentage difference in poor mental health days 14 days or more from 2014 to 2024(-)

In [12]:
# load lib
library(tidyverse)
library(glmnet)
library(glmmTMB)
library(caret)
library(DHARMa)
library(readr)
library(dplyr)
library(janitor)
# set path
library(here)
setwd(here::here())

This block is use to clean the data.

In [10]:
drop_redundant_yn_features <- function(df) {
  # Get all column names
  cols <- names(df)
  
  # Find Yes/No pairs by stripping the suffix
  yes_cols <- cols[grepl("_yes", cols)]
  no_cols  <- cols[grepl("_no",  cols)]
  
  # Get base names for each
  yes_bases <- sub("_diff$", "", sub("_yes", "", yes_cols))
  no_bases  <- sub("_diff$", "", sub("_no",  "", no_cols))
  
  # Find bases that have BOTH a Yes and No column
  paired_bases <- intersect(yes_bases, no_bases)
  
  cols_to_drop <- c()
  
  for (base in paired_bases) {
    yes_col <- yes_cols[yes_bases == base]
    no_col  <- no_cols[no_bases == base]
    
    # Count NAs in each
    yes_nas <- sum(is.na(df[[yes_col]]))
    no_nas  <- sum(is.na(df[[no_col]]))
    
    # Drop whichever has more NAs 
    # If tied drop "No" (keep "Yes")
    if (no_nas >= yes_nas) {
      cols_to_drop <- c(cols_to_drop, no_col)
    } else {
      cols_to_drop <- c(cols_to_drop, yes_col)
    }
  }
  
  cat("Dropping", length(cols_to_drop), "redundant columns:\n")
  #cat(paste(" ", cols_to_drop), sep = "\n")
  
  df[, !names(df) %in% cols_to_drop]
}

set.seed(100)

df <- read_csv("data/merged_with_svi.csv", show_col_types = FALSE) |> clean_names() 

df <- df %>% filter(!county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt", "Loving"))
df_raw <- df
df$density <- df$population*1.0/df$area_sqmi
df <- df %>% dplyr::select(-c("county", "area_sqmi"), -starts_with("m_"), -starts_with("mp_"), -starts_with("e_"), -starts_with("epl_"), -starts_with("spl_"), -starts_with("rpl_"), -starts_with("f_"))
df <- df[,colSums(is.na(df)) == 0]
df <- df %>% dplyr::select(-c("ep_minrty", "ep_hisp", "ep_afam", "ep_pov150", "ep_uninsur"))

df <- drop_redundant_yn_features(df)

index <- createDataPartition(df$outbreak, p = 0.75, list = FALSE)
df <- df[index,]

df <- df[index,] %>% dplyr::select(-c("phr", "enrollment", "population", "outbreak"))

Dropping 82 redundant columns:


This block prints the top 25 variables has highest spearman correlation with the variables of interest. The correlation is computed on the training set for consistency with model fitting. The variables are sorted by absolute value of correlation.

In [11]:
selected <- c("cve", "mammogram_past_2_yrs_40_ia_yes", "skin_cancer_no", "poor_mental_health_14_days_14_or_more_days_pct_diff", "ep_nohsdp")

cve_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$cve, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))

print(cve_cors, n = 25) 

mam_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$mammogram_past_2_yrs_40_ia_yes, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(mam_cors, n = 25)  

poor_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$poor_mental_health_14_days_14_or_more_days_pct_diff, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(poor_cors, n = 25)  

hsbp_cors <- tibble(
  variable = setdiff(names(df), selected),
  rho = sapply(setdiff(names(df), selected),
               function(v) cor(df$ep_nohsdp, df[[v]], method = "spearman", use = "pairwise.complete.obs"))
) %>%
  mutate(abs_rho = abs(rho)) %>%
  arrange(desc(abs_rho))
print(hsbp_cors, n = 25)  

# A tibble: 242 × 3
   variable                              rho abs_rho
   <chr>                               <dbl>   <dbl>
 1 pct_white                           0.624   0.624
 2 pct_hispanic                       -0.557   0.557
 3 ep_limeng                          -0.507   0.507
 4 rx_opioid_no                       -0.351   0.351
 5 pct_poverty                        -0.341   0.341
 6 ep_age65                            0.337   0.337
 7 stroke_no                          -0.328   0.328
 8 ep_twomore                          0.326   0.326
 9 ep_age17                           -0.319   0.319
10 pre_diabetes_2_yes                 -0.314   0.314
11 rx_pain_risk_ed_source_no          -0.309   0.309
12 stroke_no_pct_diff                 -0.308   0.308
13 pneumonia_shot_65_yes               0.306   0.306
14 pre_diabetes_yes                   -0.305   0.305
15 ep_noint                           -0.304   0.304
16 overweight_or_obese_yes_pct_diff    0.299   0.299
17 last_smoked_10_years   